# MedNorm-VI S1 — Mention Dataloader + Training Smoke (first Colab run)

**Colab-only.** A dataloader + training-path smoke test, NOT full training.
See docs/training/first_colab_run.md.


## 1. Environment & runtime checks


In [ ]:
import sys
import platform

print('python', platform.python_version())
print('IN_COLAB', 'google.colab' in sys.modules)
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print('gpu', torch.cuda.get_device_name(0), 'vram_gb', round(props.total_memory / 1e9, 1))
except ImportError as exc:
    print('torch not installed yet:', exc)


## 2. Google Drive configuration (edit PROJECT_ROOT only)


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/mednorm-vi'   # <-- EDIT THIS
DATA_ROOT = PROJECT_ROOT + '/data'
MODEL_CACHE = PROJECT_ROOT + '/model_cache'
CHECKPOINT_OUT = PROJECT_ROOT + '/checkpoints/full_v1/mention/vihealthbert'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError as exc:
    print('not in Colab:', exc)


## 3. Dependency installation (pinned)


In [ ]:
PINS = ['torch==2.3.1', 'transformers==4.44.2', 'datasets==2.21.0', 'accelerate==0.33.0']
# !pip -q install {' '.join(PINS)}
print('pins', PINS)


## 4. Repository checkout / expected commit


In [ ]:
EXPECTED_COMMIT = '<fill: reviewed repo HEAD>'
SEED = 20260723
print('expected_commit', EXPECTED_COMMIT, 'seed', SEED)


## 5. Data verification (governed corpus hashes)


In [ ]:
import json
import hashlib

CORPUS = DATA_ROOT + '/derived/training_corpora/mednorm_vi_training_v1'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

with open(CORPUS + '/manifests/split_manifest.json') as fh:
    split_manifest = json.load(fh)
for split, expected in split_manifest['sha256'].items():
    got = sha256(CORPUS + '/splits/' + split + '.jsonl')
    assert got == expected, split + ' hash mismatch'
    print(split, 'OK', split_manifest['counts'][split])
print('cross_split_group_leakage', split_manifest['cross_split_group_leakage'])


## 6. Model acquisition (Colab only; pinned revision; no external inference APIs)


In [ ]:
MODEL = 'demdecuong/vihealthbert-base-word'   # primary S1 backbone (verify)
MODEL_REVISION = '<pin model revision>'
# from transformers import AutoModel, AutoTokenizer
# tok = AutoTokenizer.from_pretrained(MODEL, revision=MODEL_REVISION, cache_dir=MODEL_CACHE)
# net = AutoModel.from_pretrained(MODEL, revision=MODEL_REVISION, cache_dir=MODEL_CACHE)
print('model', MODEL, 'revision', MODEL_REVISION)  # base-model weights are never committed


## 7. Training configuration


In [ ]:
CONFIG = {
    'seed': SEED, 'batch_size': 8, 'grad_accum': 2, 'mixed_precision': 'bf16',
    'checkpoint_every': 50, 'early_stopping_patience': 3, 'resume': True,
    'deterministic': True, 'max_smoke_batches': 3,
}
print(CONFIG)


## 8. Smoke mode (a few batches; verifies the path, NOT a trained model)


In [ ]:
SMOKE = True
# Build BIO features from splits/train.jsonl and run CONFIG['max_smoke_batches']
# forward/backward steps using the tok/net above; assert the loss is finite.
print('smoke would run', CONFIG['max_smoke_batches'], 'batches')


## 9. Full mode (OFF by default; must be enabled deliberately)


In [ ]:
RUN_FULL_TRAINING = False   # never auto-runs
if RUN_FULL_TRAINING:
    raise SystemExit('Full training is user-gated; enable only after a good smoke run.')
print('full_training_disabled')


## 10. Artifact export (smoke checkpoint + manifest)


In [ ]:
import os

os.makedirs(CHECKPOINT_OUT, exist_ok=True)
manifest = {
    'manifest_version': 1, 'role': 'mention/vihealthbert',
    'base_model': {'name': 'ViHealthBERT', 'revision': MODEL_REVISION},
    'training': {'git_commit': EXPECTED_COMMIT, 'seed': SEED, 'mode': 'smoke',
                 'dataset_manifest_hash': '<fill>', 'split_manifest_hash': '<fill>'},
    'status': 'SMOKE_ONLY',
}
with open(CHECKPOINT_OUT + '/checkpoint_manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('wrote smoke manifest ->', CHECKPOINT_OUT)


## 11. Resume & failure recovery


In [ ]:
# On restart: detect the latest valid checkpoint under CHECKPOINT_OUT and resume.
# Never overwrite a better checkpoint silently (compare metrics before replacing).
print('resume policy: keep best-by-metric; save every checkpoint_every steps')


## 12. Return to repo
Copy CHECKPOINT_OUT into models/checkpoints/full_v1/mention/vihealthbert/ (weights
git-ignored; manifest reviewable), then run the validation commands in
docs/training/first_colab_run.md. Do NOT commit restricted base-model weights.
